In [1]:
#!/usr/bin/env python3
"""
PII Scanner - 逐行處理版
支援大型檔案分段/逐行掃描，降低記憶體負擔並修正存檔邏輯
"""

import spacy
from pathlib import Path
import time
import shutil
from pathlib import Path

# ── 設定 ────────────────────────────────────────────────────────
MODEL_PATH = "./output/model-best-2"  # 改成你的模型路徑

MASK = {
    "PERSON":        "[PERSON_REDACTED]",
    "API_KEY":       "[API_KEY_REDACTED]",
    "EMAIL_ADDRESS": "[EMAIL_REDACTED]",
    "PHONE_NUMBER":  "[PHONE_REDACTED]",
    "IP_ADDRESS":    "[IP_REDACTED]",
    "CREDIT_CARD":   "[CREDIT_CARD_REDACTED]",
    "US_SSN":        "[SSN_REDACTED]",
    "IBAN_CODE":     "[IBAN_REDACTED]",
    "LOCATION":      "[LOCATION_REDACTED]",
}

IGNORE_LABELS = {"DATE_TIME"}


def scan_and_redact_line(line_text, nlp, line_offset=0):
    """
    針對單一行文字進行 NER 辨識與遮蔽
    line_offset: 該行在整個檔案中的起始字元位置，用於計算全域 start/end
    """
    doc = nlp(line_text)
    found_in_line = []
    redacted_line = line_text

    # 由後往前取代，避免字元索引（index）因字串長度改變而位移
    for ent in sorted(doc.ents, key=lambda e: e.start_char, reverse=True):
        if ent.label_ in IGNORE_LABELS:
            continue

        mask = MASK.get(ent.label_, f"[{ent.label_}_REDACTED]")
        
        found_in_line.append({
            "text":  ent.text,
            "label": ent.label_,
            "start": line_offset + ent.start_char,
            "end":   line_offset + ent.end_char,
        })

        redacted_line = (
            redacted_line[:ent.start_char] +
            mask +
            redacted_line[ent.end_char:]
        )

    # 由於是反向 append，這裡 reverse 回正向順序
    found_in_line.reverse()
    return found_in_line, redacted_line


def main():
    # ── 載入模型 ────────────────────────────────────────────────
    print(f"載入模型：{MODEL_PATH}")
    try:
        nlp = spacy.load(MODEL_PATH)
        print("✅ 模型載入成功")
        print(f"   支援實體：{nlp.get_pipe('ner').labels}\n")
    except Exception as e:
        print(f"❌ 模型載入失敗：{e}")
        return

  
    start_time = time.perf_counter()
    root = r"D:\2.programm2\Exam\commission3"
    folder = Path(root)
    
    for file in folder.rglob("*"):
        if file.is_file() and file.suffix.lower() in {".txt", ".ipynb", ".csv", ".json", ".xml", ".yml", ".yaml", ".md", ".py", ".js", ".ts", ".java", ".c", ".cpp", ".h", ".ini", ".conf", ".env", ".sql", ".html", ".htm", ".sh"}:
            all_found = []
            redacted_lines = []
            current_offset = 0
            with file.open("r", encoding="utf-8", errors="ignore") as f:
                for line in f:
                    found_in_line, redacted_line = scan_and_redact_line(
                        line, nlp, line_offset=current_offset
                    )
                    all_found.extend(found_in_line)
                    redacted_lines.append(redacted_line)
                    
                    # 累加當前行的字元長度（含換行符號）
                    current_offset += len(line)
            
            print(f"發現 {len(all_found)} 個敏感實體：\n")
            for i, ent in enumerate(all_found, 1):
                print(f"  [{i}] {ent['label']:<20} (pos: {ent['start']}-{ent['end']}) → {ent['text']!r}")
    end_time = time.perf_counter()
    print(f"\n總共耗時: {end_time - start_time:.2f} 秒")



if __name__ == "__main__":
    main()

In [4]:
from ckip_transformers.nlp import CkipWordSegmenter, CkipPosTagger, CkipNerChunker

# 1. 初始化模型驅動器（預設會自動從 Hugging Face 下載 checkpoint）
# device=-1 代表使用 CPU，若有 GPU 可設為 device=0
ws_driver = CkipWordSegmenter(model="bert-base", device=-1)
pos_driver = CkipPosTagger(model="bert-base", device=-1)
ner_driver = CkipNerChunker(model="bert-base", device=-1)

# 2. 準備輸入文本
texts = [
    "中央研究院位於台北市南港區，是台灣最高學術研究機構。",
    "今天天氣真好，想去公園散步。"
]

# 3. 執行分析管道 (Pipeline)
# (1) 中文斷詞：輸入為句子列表 list[str]
ws_results = ws_driver(texts)

# (2) 詞性標記：輸入為斷詞後的列表 list[list[str]]
pos_results = pos_driver(ws_results)

# (3) 命名實體辨識：輸入為句子列表 list[str]
ner_results = ner_driver(texts)

# 4. 印出第一個句子的斷詞與實體辨識結果
print("斷詞結果：", ws_results[0])
print("詞性結果：", pos_results[0])
print("NER 結果：", ner_results[0])

'(ProtocolError('Connection aborted.', ConnectionResetError(10054, '遠端主機已強制關閉一個現存的連線。', None, 10054, None)), '(Request ID: 059bfc9b-2f19-4c3c-9e03-c8c42e21c89f)')' thrown while requesting HEAD https://huggingface.co/ckiplab/bert-base-chinese-ws/resolve/main/config.json
Retrying in 1s [Retry 1/5].
'(ProtocolError('Connection aborted.', ConnectionResetError(10054, '遠端主機已強制關閉一個現存的連線。', None, 10054, None)), '(Request ID: 1af3df42-1cd4-49b4-a1fc-6469b82c3b8b)')' thrown while requesting HEAD https://huggingface.co/ckiplab/bert-base-chinese-ws/resolve/main/config.json
Retrying in 2s [Retry 2/5].
Inference: 100%|██████████| 1/1 [00:00<00:00,  8.82it/s]

斷詞結果： ['中央', '研究院', '位於', '台北市', '南港區', '，', '是', '台灣', '最', '高', '學術', '研究', '機構', '。']
詞性結果： ['Nc', 'Nc', 'VCL', 'Nc', 'Nc', 'COMMACATEGORY', 'SHI', 'Nc', 'Dfa', 'VH', 'Na', 'VE', 'Na', 'PERIODCATEGORY']
NER 結果： [NerToken(word='中央研究院', ner='ORG', idx=(0, 5)), NerToken(word='台北市南港區', ner='LOC', idx=(7, 13)), NerToken(word='台灣', ner='GPE', idx=(15, 17))]
